# Lab 3 · RBVR · Risk-Based Vulnerability Remediation
**AI 4 Security · Parte 2 · AI per la Cyber Security** — Versione studente

Uno scanner ha prodotto decine di vulnerabilità su una dozzina di server. Aprire un ticket per ciascuna è ingestibile.
Obiettivo: arrivare a un **piano operativo**: *"applica N azioni ed elimini l'80% del rischio"*, con un report leggibile dal management.

| Passo | Cosa fai | Modulo 15 |
|---|---|---|
| 1 · Rischio | `rischio = CVSS × EPSS × criticità dell'asset` | risk scoring |
| 2 · Normalizza | etichette canoniche delle remediation: regex, poi LLM con guardrail | aggregazione con LLM |
| 3 · Aggrega | greedy set cover pesato per rischio, curva di Pareto | ottimizzazione |
| 4 · Piano e report | team ed effort, report per il management, verifica dei numeri | co-pilota |
| 5 · La tua sfida | caso avversario, trappola anti-merge, determinismo, schema, ticket, fact-checking | guardrail |

**Consegna:** il piano (N azioni, % di rischio eliminato), il report executive e la verifica dei numeri scritti dall'LLM.

## 0 · Preparazione

In [ ]:
%pip install -q google-genai pydantic

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", 60)
plt.rcParams["figure.figsize"] = (8, 4)
print("Ambiente pronto")

In [ ]:
# @title Generazione del report dello scanner (eseguire senza modificare)
import numpy as np, pandas as pd

# Azioni di remediation canoniche: prodotto, versione di destinazione, vulnerabilità chiuse, modi diversi di scriverle.
# CVE reali per Log4j e OpenSSL 3.0.x (CVSS v3 ufficiali); gli ID "SIM-" sono SINTETICI. Gli EPSS sono valori di esempio.
CATALOGO = {
    "UPGRADE · log4j-core · 2.17.1": dict(
        vulns={"CVE-2021-44228": (10.0, 0.94), "CVE-2021-45046": (9.0, 0.60), "CVE-2021-45105": (5.9, 0.35), "CVE-2021-44832": (6.6, 0.20)},
        testi=["Upgrade log4j to 2.17.1", "Aggiornare Log4J (>=2.17.1)", "log4j-core: bump to v2.17.1",
               "patch log4j 2.17.1 asap!!", "Apache Log4j2 update 2.17.1", "upd l4j -> 2.17.1"]),
    "UPGRADE · openssl · 3.0.7": dict(
        vulns={"CVE-2022-3602": (7.5, 0.08), "CVE-2022-3786": (7.5, 0.06)},
        testi=["Upgrade OpenSSL to 3.0.7", "openssl 3.0.x -> 3.0.7", "Aggiornare OpenSSL alla 3.0.7",
               "OpenSSL update (3.0.7)", "ossl upgrade 3.0.7"]),
    "UPGRADE · openssl · 1.1.1w": dict(
        vulns={"SIM-2023-0101": (7.4, 0.12), "SIM-2023-0102": (5.3, 0.03)},
        testi=["Upgrade OpenSSL to 1.1.1w", "openssl 1.1.1 -> 1.1.1w", "Aggiornare OpenSSL alla 1.1.1w", "ossl upgrade 1.1.1w"]),
    "UPGRADE · nginx · 1.24.0": dict(
        vulns={"SIM-2023-0201": (7.5, 0.30), "SIM-2023-0202": (5.3, 0.05), "SIM-2023-0203": (6.1, 0.02)},
        testi=["Upgrade nginx to 1.24.0", "nginx >= 1.24.0", "Aggiornamento NGINX 1.24.0", "update nginx pkg (1.24.0)"]),
    "UPGRADE · openssh · 9.6p1": dict(
        vulns={"SIM-2023-0301": (5.9, 0.45), "SIM-2023-0302": (6.5, 0.05)},
        testi=["Upgrade OpenSSH to 9.6p1", "openssh-server 9.6p1", "Aggiornare SSH (OpenSSH 9.6p1)", "sshd update 9.6p1"]),
    "UPGRADE · httpd · 2.4.58": dict(
        vulns={"SIM-2023-0401": (9.8, 0.55), "SIM-2023-0402": (7.5, 0.10), "SIM-2023-0403": (5.3, 0.01)},
        testi=["Upgrade Apache HTTP Server to 2.4.58", "httpd 2.4.58", "Aggiornare Apache httpd alla 2.4.58", "apache2 update -> 2.4.58"]),
    "UPGRADE · jquery · 3.5.0": dict(
        vulns={"SIM-2020-0501": (6.1, 0.02), "SIM-2020-0502": (6.1, 0.01)},
        testi=["Upgrade jQuery to 3.5.0", "jquery.js >= 3.5.0", "Aggiornare la libreria jQuery (3.5.0)"]),
    "UPGRADE · kernel · 6.1.70": dict(
        vulns={"SIM-2023-0601": (7.8, 0.15), "SIM-2023-0602": (7.0, 0.04), "SIM-2023-0603": (5.5, 0.01)},
        testi=["Apply kernel patch 6.1.70", "Linux kernel update 6.1.70", "Aggiornare il kernel alla 6.1.70",
               "kernel-core 6.1.70 (reboot required)"]),
    "UPGRADE · tomcat · 9.0.83": dict(
        vulns={"SIM-2023-0701": (7.5, 0.25), "SIM-2023-0702": (5.3, 0.03)},
        testi=["Upgrade Apache Tomcat to 9.0.83", "tomcat9 9.0.83", "Aggiornare Tomcat alla 9.0.83"]),
    "CONFIG · tls · disabilita 1.0/1.1": dict(
        vulns={"SIM-2019-0801": (5.9, 0.01)},
        testi=["Disable TLS 1.0/1.1", "Disabilitare TLS 1.0 e 1.1 nella configurazione", "tls config: min protocol TLSv1.2"]),
}

ASSET = pd.DataFrame([
    ("web-01", "frontend e-commerce", 3, "web"), ("web-02", "frontend e-commerce", 3, "web"),
    ("web-03", "sito istituzionale", 1, "web"),
    ("app-01", "backend ordini", 3, "java-container"), ("app-02", "backend ordini", 3, "java-container"),
    ("app-03", "backend reportistica", 2, "java-container"), ("api-01", "API partner", 3, "java-container"),
    ("legacy-01", "gestionale storico", 2, "legacy-rhel7"), ("legacy-02", "gestionale storico", 2, "legacy-rhel7"),
    ("db-01", "database clienti", 3, "db"), ("bastion-01", "accesso amministrativo", 3, "bastion"),
    ("test-01", "ambiente di test", 1, "java-container"),
], columns=["asset", "ruolo", "criticita", "piattaforma"])

# quali prodotti (azioni canoniche) sono presenti su ciascuna piattaforma
PRESENZA = {
    "web": ["UPGRADE · nginx · 1.24.0", "UPGRADE · openssl · 3.0.7", "UPGRADE · jquery · 3.5.0", "UPGRADE · kernel · 6.1.70"],
    "java-container": ["UPGRADE · log4j-core · 2.17.1", "UPGRADE · openssl · 3.0.7", "UPGRADE · tomcat · 9.0.83"],
    "legacy-rhel7": ["UPGRADE · openssl · 1.1.1w", "UPGRADE · httpd · 2.4.58", "UPGRADE · openssh · 9.6p1",
                     "CONFIG · tls · disabilita 1.0/1.1", "UPGRADE · kernel · 6.1.70"],
    "db": ["UPGRADE · kernel · 6.1.70", "UPGRADE · openssh · 9.6p1"],
    "bastion": ["UPGRADE · openssh · 9.6p1", "UPGRADE · kernel · 6.1.70"],
}

# azioni "di piattaforma": una sola operazione chiude tutti i finding di certi prodotti su una piattaforma
PIATTAFORMA = {
    "REBUILD · immagine base-java · 2024.06": dict(piattaforma="java-container",
        copre=["UPGRADE · log4j-core · 2.17.1", "UPGRADE · openssl · 3.0.7", "UPGRADE · tomcat · 9.0.83"], effort=24),
    "MIGRATE · legacy RHEL7 → RHEL9": dict(piattaforma="legacy-rhel7",
        copre=PRESENZA["legacy-rhel7"], effort=80),
}

EFFORT = {"log4j-core": 6, "openssl": 4, "nginx": 3, "openssh": 3, "httpd": 5, "jquery": 2, "kernel": 6, "tomcat": 5, "tls": 2}


def genera_report_scanner(seed=11):
    """Report sintetico di uno scanner di vulnerabilità: una riga per (asset, vulnerabilità)."""
    rng = np.random.default_rng(seed)
    righe, verita = [], []
    for _, a in ASSET.iterrows():
        for can in PRESENZA[a["piattaforma"]]:
            info = CATALOGO[can]
            for vid, (cvss, epss) in info["vulns"].items():
                if rng.random() < 0.15:          # non tutti gli asset hanno tutte le vulnerabilità
                    continue
                testo = info["testi"][rng.integers(len(info["testi"]))]
                righe.append((a["asset"], a["criticita"], vid, cvss, epss, testo))
                verita.append(can)
    df = pd.DataFrame(righe, columns=["asset", "criticita", "vuln_id", "cvss", "epss", "remediation"])
    df.insert(0, "finding_id", [f"F{i:03d}" for i in range(len(df))])
    return df, pd.Series(verita, name="canonica")

print('Generatore pronto')

In [ ]:
# @title Configurazione LLM (Gemini se disponibile, altrimenti simulazione offline)
MODELLO = "gemini-2.5-flash"
client, types = None, None
try:
    from google import genai
    from google.genai import types
    try:
        from google.colab import userdata
        chiave = userdata.get("GEMINI_API_KEY")
    except Exception:
        import os
        chiave = os.environ.get("GEMINI_API_KEY")
    if chiave:
        client = genai.Client(api_key=chiave)
except Exception as e:
    print("Gemini non disponibile:", type(e).__name__)


def chiedi_llm(prompt, schema, simulazione, temperature=0.0):
    """Gemini con output JSON validato dallo schema Pydantic; altrimenti simulazione offline."""
    if client is not None:
        try:
            r = client.models.generate_content(
                model=MODELLO, contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature,
                                                   response_mime_type="application/json",
                                                   response_schema=schema))
            return schema.model_validate_json(r.text)
        except Exception as e:
            print("Errore Gemini, uso la simulazione:", type(e).__name__)
    return schema.model_validate(simulazione(prompt))

print("Modalità LLM:", f"Gemini ({MODELLO})" if client else "SIMULAZIONE OFFLINE")

Per usare Gemini: crea una chiave su Google AI Studio e aggiungila nei **Secrets** di Colab (icona della chiave) con il nome `GEMINI_API_KEY`.
Senza chiave il laboratorio usa una **simulazione offline** e lo dichiara in ogni output.

---
## 1 · Il report dello scanner e il rischio

Ogni riga è un **finding**: una vulnerabilità (`vuln_id`) su un asset, con il suo CVSS, il suo EPSS e il testo di remediation
scritto dallo scanner. La tabella `ASSET` descrive i server e la loro criticità (1 = bassa, 3 = alta).

In [ ]:
report, verita = genera_report_scanner()      # `verita` = etichetta canonica corretta, solo per misurare
print("Finding:", len(report), "· vulnerabilità distinte:", report["vuln_id"].nunique(),
      "· testi di remediation distinti:", report["remediation"].nunique())
display(ASSET)
report.head(10)

**Domanda:** 77 finding e 34 testi di remediation diversi. Quante azioni diverse pensi ci siano davvero?
Scorri la colonna `remediation` e prova a stimarlo.

*Risposta:* 

### 1.1 · Pesare il rischio

- **CVSS** (0-10): gravità tecnica.
- **EPSS** (0-1): probabilità che la vulnerabilità venga sfruttata nei prossimi 30 giorni.
- **Criticità dell'asset** (1-3): quanto ci costa se quel sistema viene compromesso.

In [ ]:
# TODO: crea la colonna rischio = cvss * epss * criticita e mostra i 10 finding più rischiosi
report["rischio"] = ...

In [ ]:
# Controllo
assert abs(report["rischio"].sum() - 273.08) < 0.1, "Il rischio totale atteso è circa 273,1"
print("OK")

### 1.2 · CVSS da solo basta? Confronta le due classifiche

In [ ]:
# TODO: confronta le 10 righe con CVSS più alto e le 10 con rischio più alto (nlargest): quante sono in comune?
#       Poi disegna uno scatter CVSS (x) vs EPSS (y) con dimensione = criticita*40 e colore = rischio

---
## 2 · Normalizzare le remediation

Per aggregare, due testi che descrivono la **stessa azione** devono avere la **stessa etichetta canonica**, nel formato:

`AZIONE · prodotto · versione`, per esempio `UPGRADE · log4j-core · 2.17.1` oppure `CONFIG · tls · disabilita 1.0/1.1`.

Misureremo ogni metodo con l'**accuratezza di normalizzazione**: la quota di finding a cui è assegnata l'etichetta corretta (`verita`).

In [ ]:
def accuratezza(etichette):
    return (pd.Series(etichette, index=report.index) == verita).mean()

testi = sorted(report["remediation"].unique())
print(len(testi), "testi distinti, per esempio:")
print("\n".join(testi[:12]))

### 2.1 · Primo tentativo: una regex

In [ ]:
PRODOTTI = {"log4j": "log4j-core", "openssl": "openssl", "nginx": "nginx", "openssh": "openssh",
            "httpd": "httpd", "jquery": "jquery", "kernel": "kernel", "tomcat": "tomcat"}
RE_PROD = re.compile("|".join(PRODOTTI), re.I)
RE_VER = re.compile(r"\d+\.\d+(?:\.\d+)?[a-z0-9]*", re.I)

def normalizza_regex(testo):
    # TODO: cerca il prodotto con RE_PROD e tutte le versioni con RE_VER.findall
    #       se manca uno dei due restituisci "NON RICONOSCIUTA"
    #       altrimenti "UPGRADE · <prodotto canonico> · <ultima versione trovata>"
    ...

report["azione_regex"] = report["remediation"].map(normalizza_regex)
print("Accuratezza regex:", round(accuratezza(report["azione_regex"]), 3))

### 2.2 · Normalizzare con un LLM

Chiediamo all'LLM di assegnare a **ogni testo distinto** un'etichetta canonica, con un output JSON garantito da uno schema.
Un buon prompt fissa il formato con **esempi** (modulo 8).

In [ ]:
class Voce(BaseModel):
    testo: str = Field(description="il testo originale, copiato identico")
    azione: str = Field(description="UPGRADE oppure CONFIG")
    prodotto: str = Field(description="nome canonico minuscolo, es. log4j-core, openssl, httpd, openssh, tls")
    versione: str = Field(description="versione di destinazione; per CONFIG una breve descrizione")

class Normalizzazione(BaseModel):
    voci: list[Voce]


ALIAS = [("tomcat", "tomcat"), ("log4j", "log4j-core"), ("l4j", "log4j-core"), ("openssh", "openssh"), ("sshd", "openssh"),
         ("openssl", "openssl"), ("ossl", "openssl"), ("nginx", "nginx"), ("httpd", "httpd"), ("apache http", "httpd"),
         ("apache2", "httpd"), ("jquery", "jquery"), ("kernel", "kernel")]

def simula_normalizzazione(prompt):
    """SIMULAZIONE OFFLINE di un LLM: conosce sinonimi e abbreviazioni, ma commette un errore plausibile
    (confonde OpenSSL 1.1.1w con 3.0.7 in un testo abbreviato), come può fare un modello reale."""
    blocco = prompt.split("<testi>")[1].split("</testi>")[0]
    voci = []
    for t in [r.strip()[2:] for r in blocco.strip().split("\n") if r.strip().startswith("- ")]:
        low = t.lower()
        if "tls" in low:
            voci.append(dict(testo=t, azione="CONFIG", prodotto="tls", versione="disabilita 1.0/1.1")); continue
        prod = next((c for a, c in ALIAS if a in low), "sconosciuto")
        ver = RE_VER.findall(t)
        ver = ver[-1] if ver else "?"
        if t == "ossl upgrade 1.1.1w":
            ver = "3.0.7"                                      # errore voluto della simulazione
        voci.append(dict(testo=t, azione="UPGRADE", prodotto=prod, versione=ver))
    return {"voci": voci}

In [ ]:
def prompt_normalizzazione(testi):
    elenco = "\n".join(f"- {t}" for t in testi)
    # TODO: scrivi il prompt con ruolo, compito, regole sui nomi canonici (log4j-core, httpd, openssh, tls),
    #       il divieto di unire prodotti o versioni diversi, due esempi,
    #       e i testi racchiusi fra <testi> e </testi> (la simulazione offline li cerca lì)
    return f"""...
<testi>
{elenco}
</testi>"""

risposta = chiedi_llm(prompt_normalizzazione(testi), Normalizzazione, simula_normalizzazione)
# TODO: costruisci mappa_llm: testo -> "AZIONE · prodotto · versione" e applicala alla colonna remediation
mappa_llm = ...
report["azione_llm"] = ...
print("Accuratezza LLM:", round(accuratezza(report["azione_llm"]), 3))

### 2.3 · Il guardrail: validare prima di fidarsi

L'LLM generalizza, ma può sbagliare in modo **plausibile**. Un merge sbagliato è pericoloso: se due azioni diverse finiscono sotto la
stessa etichetta, applicheremo una sola fix e **lasceremo CVE aperte**.

Scrivi `valida(testo, etichetta)` che accetta un'etichetta solo se:
1. per le azioni `UPGRADE` la **versione** compare letteralmente nel testo originale;
2. il **prodotto** è uno di quelli ammessi.

Le etichette che non passano vanno a **revisione umana**.

In [ ]:
AMMESSI = {"log4j-core", "openssl", "nginx", "openssh", "httpd", "jquery", "kernel", "tomcat", "tls"}

def valida(testo, etichetta):
    # TODO: restituisci True solo se l'etichetta ha 3 parti, il prodotto è in AMMESSI
    #       e, per le azioni UPGRADE, la versione compare nel testo originale
    ...

report["valida"] = [valida(t, e) for t, e in zip(report["remediation"], report["azione_llm"])]
da_rivedere = report.loc[~report["valida"], ["remediation", "azione_llm"]].drop_duplicates()
print("Etichette respinte dal guardrail:", len(da_rivedere))
da_rivedere

Simuliamo la **revisione umana**: l'analista corregge a mano le etichette respinte (qui usiamo la verità, come farebbe un esperto).
Poi confrontiamo le tre accuratezze.

In [ ]:
report["azione"] = np.where(report["valida"], report["azione_llm"], verita)
confronto = pd.Series({"regex": accuratezza(report["azione_regex"]),
                       "LLM": accuratezza(report["azione_llm"]),
                       "LLM + guardrail + revisione": accuratezza(report["azione"])}).round(3)
print(confronto)
print("\nAzioni distinte dopo la normalizzazione:", report["azione"].nunique())

---
## 3 · Aggregare: il greedy set cover

Ogni **azione** chiude un insieme di finding. Oltre alle azioni del report esistono due **azioni di piattaforma** (dizionario `PIATTAFORMA`):
ricostruire l'immagine base dei container Java, o migrare i server legacy. Una sola operazione chiude molti finding insieme.

Costruiamo `copertura`: per ogni azione, l'insieme dei `finding_id` che chiude.

In [ ]:
piattaforma_di = dict(zip(ASSET["asset"], ASSET["piattaforma"]))
report["piattaforma"] = report["asset"].map(piattaforma_di)

copertura = {a: set(g["finding_id"]) for a, g in report.groupby("azione")}
for nome, info in PIATTAFORMA.items():
    k = (report["piattaforma"] == info["piattaforma"]) & report["azione"].isin(info["copre"])
    copertura[nome] = set(report.loc[k, "finding_id"])

rischio = report.set_index("finding_id")["rischio"]
pd.Series({a: (len(f), round(rischio[list(f)].sum(), 1)) for a, f in copertura.items()},
          name="(finding, rischio)").to_frame()

### 3.1 · L'algoritmo greedy

A ogni passo scegli l'azione che copre il **rischio residuo maggiore** (cioè il rischio dei finding non ancora coperti),
aggiungila al piano e segna come coperti i suoi finding. Fermati quando hai coperto la quota `obiettivo` del rischio totale.

In [ ]:
def greedy_set_cover(copertura, rischio, obiettivo=0.8, costo=None):
    totale = rischio.sum()
    coperti, piano, cumulato = set(), [], 0.0

    def guadagno(a):
        # TODO: rischio dei finding di `a` non ancora coperti (diviso per costo[a] se costo è dato)
        ...

    while cumulato < obiettivo * totale - 1e-9:
        # TODO: scegli l'azione con guadagno massimo, aggiorna coperti e cumulato,
        #       aggiungi a `piano` un dizionario con passo, azione, finding_chiusi, rischio_coperto, cumulato_%
        ...
    return pd.DataFrame(piano)

piano = greedy_set_cover(copertura, rischio, obiettivo=0.8)
piano

In [ ]:
# Controllo
assert piano["cumulato_%"].iloc[-1] >= 80 and piano["cumulato_%"].is_monotonic_increasing
print(f"Applica {len(piano)} azioni: {piano['cumulato_%'].iloc[-1]:.0f}% del rischio eliminato "
      f"({piano['finding_chiusi'].sum()} finding su {len(report)})")

### 3.2 · La curva di Pareto

In [ ]:
completo = greedy_set_cover(copertura, rischio, obiettivo=1.0)
plt.plot(completo["passo"], completo["cumulato_%"], marker="o", color="#0A0AB4")
plt.axhline(80, color="#E3002B", ls="--")
plt.xlabel("azioni applicate (ordine greedy)"); plt.ylabel("% rischio eliminato"); plt.ylim(0, 105)
plt.title("Poche azioni, molto rischio"); plt.show()
completo[["passo", "azione", "finding_chiusi", "cumulato_%"]].round(1)

### 3.3 · E se contasse anche l'effort?

La migrazione dei server legacy vale molto rischio ma costa 80 ore. Ripeti il greedy dividendo il guadagno per l'effort
(`costo`, in ore) e confronta i due piani.

In [ ]:
# TODO: costruisci il dizionario costo (azione -> ore) usando PIATTAFORMA[...]["effort"] per le azioni di piattaforma
#       ed EFFORT[prodotto] per le altre (il prodotto è la seconda parte dell'etichetta), poi rifai il greedy con costo=costo
costo = ...
piano_eff = ...

---
## 4 · Piano operativo e report con l'LLM come co-pilota

L'LLM **arricchisce** il piano (team responsabile, note operative) e **racconta** i numeri al management.
Regola: **i numeri li calcola il codice**, il modello si limita a raccontarli.

In [ ]:
class Assegnazione(BaseModel):
    azione: str
    team: str = Field(description="uno fra: Middleware, Sistemi, Web, Sicurezza, Piattaforme container")
    nota: str = Field(description="una frase operativa: finestra di manutenzione, test, rollback")

class Piano(BaseModel):
    assegnazioni: list[Assegnazione]

TEAM = {"log4j-core": "Middleware", "tomcat": "Middleware", "openssl": "Sistemi", "kernel": "Sistemi", "openssh": "Sistemi",
        "nginx": "Web", "httpd": "Web", "jquery": "Web", "tls": "Sicurezza"}

def simula_piano(prompt):
    righe = prompt.split("<azioni>")[1].split("</azioni>")[0].strip().split("\n")
    out = []
    for r in righe:
        a = r[2:]
        if a.startswith("REBUILD") or a.startswith("MIGRATE"):
            team = "Piattaforme container" if "java" in a else "Sistemi"
        else:
            team = TEAM.get(a.split(" · ")[1], "Sicurezza")
        out.append(dict(azione=a, team=team, nota="pianificare in finestra di manutenzione con test e piano di rollback"))
    return {"assegnazioni": out}

In [ ]:
elenco = "\n".join(f"- {a}" for a in piano["azione"])
# TODO: scrivi prompt_piano (ruolo di coordinatore, elenco dei team ammessi, azioni fra <azioni> e </azioni>),
#       chiama chiedi_llm con lo schema Piano e unisci il risultato al DataFrame `piano`
prompt_piano = ...

### 4.1 · Il report per il management: i numeri li calcola il codice

In [ ]:
class Report(BaseModel):
    titolo: str
    sintesi: str = Field(description="3-4 frasi per un pubblico non tecnico")
    punti: list[str] = Field(description="3-5 punti elenco con i numeri forniti, senza inventarne altri")

numeri = {
    "finding_totali": len(report),
    "azioni_nel_piano": len(piano),
    "rischio_eliminato_%": round(piano["cumulato_%"].iloc[-1], 1),
    "finding_chiusi": int(piano["finding_chiusi"].sum()),
    "ore_stimate": int(piano["ore"].sum()),
    "prima_azione": piano["azione"].iloc[0],
}

def simula_report(prompt):
    n = numeri
    return dict(titolo="Piano di remediation basato sul rischio",
                sintesi=(f"Lo scanner ha rilevato {n['finding_totali']} vulnerabilità sui nostri sistemi. Invece di correggerle una per una, "
                         f"proponiamo {n['azioni_nel_piano']} interventi che eliminano il {n['rischio_eliminato_%']}% del rischio stimato. "
                         f"L'impegno previsto è di circa {n['ore_stimate']} ore."),
                punti=[f"Prima azione: {n['prima_azione']}", f"Vulnerabilità chiuse: {n['finding_chiusi']} su {n['finding_totali']}",
                       f"Rischio eliminato: {n['rischio_eliminato_%']}%"])

prompt_report = f"""Scrivi un breve report per il management sul piano di remediation.
Usa SOLO questi numeri, senza calcolarne o inventarne altri: {numeri}"""
rep = chiedi_llm(prompt_report, Report, simula_report)
print(rep.titolo.upper(), "\n\n" + rep.sintesi + "\n")
print("\n".join("- " + p for p in rep.punti))

### 4.2 · Verifica automatica: ogni numero nel report deve essere uno dei nostri

In [ ]:
# TODO: estrai con una regex tutti i numeri dal testo del report (sintesi + punti) e segnala
#       quelli che non compaiono fra i valori del dizionario `numeri` (tolleranza per le cifre delle etichette di versione)

---
## 5 · La tua sfida

Scegli almeno **tre** esercizi. Per ognuno annota il risultato in una riga.

### 5.1 · Caso avversario

Aggiungi testi "cattivi": abbreviazioni, lingue miste, refusi (per esempio `"URGENTE: aggiornare lo4j a 2.17.1"`,
`"OpenSSL: please upgrade a 3.0.7 (vedi advisory)"`, `"nginx 1.24 (.0)"`). L'LLM regge dove la regex si spezza? E il guardrail?

In [ ]:
avversari = ["URGENTE: aggiornare lo4j a 2.17.1", "OpenSSL: please upgrade a 3.0.7 (vedi advisory)",
             "nginx 1.24 (.0)", "Aggiorna tomcat9 -> 9.0.83 / riavviare", "SSH: usare openssh 9.6p1"]
# TODO: normalizza questi testi con la regex e con l'LLM e verifica ciascuna etichetta con valida()

### 5.2 · Trappola anti-merge

Verifica che il tuo guardrail **non** permetta di unire `"Upgrade OpenSSL to 3.0.7"` e `"Upgrade OpenSSL to 1.1.1w"` sotto la stessa etichetta.
Scrivi un test con `assert`.

In [ ]:
# TODO: scrivi tre assert: uno che accetta l'etichetta giusta e due che rifiutano il merge 1.1.1w -> 3.0.7

### 5.3 · Determinismo

Con Gemini: normalizza gli stessi testi 3 volte con `temperature=0` e 3 volte con `temperature=1`. Quante etichette cambiano fra le
esecuzioni? (In simulazione offline l'output è sempre identico: l'esercizio richiede la chiave API.)

In [ ]:
# TODO: esegui la normalizzazione 3 volte per temperature 0.0 e 1.0 e conta i testi con etichette diverse fra le esecuzioni

### 5.4 · Schema garantito

Rendi lo schema più stretto: il campo `azione` accetta solo `"UPGRADE"` o `"CONFIG"` e `prodotto` solo i valori di `AMMESSI`
(suggerimento: `typing.Literal`). Cosa succede se l'LLM restituisce un valore fuori elenco?

In [ ]:
# TODO: definisci VoceStretta con Literal per azione e prodotto e prova a validare un esempio sbagliato

### 5.5 · Ticket e report
Genera un ticket testuale per ogni azione del piano (titolo, team, finding chiusi, asset coinvolti) e un secondo report, **tecnico**, oltre a quello executive.

In [ ]:
# TODO: per ogni riga di piano_op stampa un ticket con azione, team, numero di finding, rischio coperto e asset coinvolti

### 5.6 · Fact-checking

Chiedi all'LLM di **stimare** quante azioni servono per eliminare l'80% del rischio **senza** dargli i dati calcolati,
poi confronta con il numero vero. Perché i numeri li deve calcolare il codice?

In [ ]:
# TODO: chiedi all'LLM una stima (schema con azioni_stimate e motivazione) senza fornire i dati calcolati
#       e confrontala con len(piano)

---
## Consegna

**Il piano:** applica ___ azioni ed elimini il ___% del rischio (___ finding su ___, circa ___ ore).

**Accuratezza di normalizzazione:** regex ___ · LLM ___ · LLM + guardrail ___

**Il report executive** (incolla qui il testo generato e segnala se la verifica dei numeri ha trovato valori estranei):

**Esercizi della sfida svolti e cosa hai imparato (una riga ciascuno):**